# Hydrology features for XGBoost

Start from the **original preprocessed CSVs**. Keep the scheduling notebook's 262 features, restore **18 reservoir-specific hourly inflow inputs**, and add **nine local water-balance proxies**: **289 features in total**.

Run on CPU with NumPy and pandas. In Colab, upload these five files together:

- `hourly_train.csv`
- `hourly_test.csv`
- `cases.csv`
- `meta.json`
- `Historical_inflow_1958_2025.csv`

The extra inflow file is in the project's `data/kernel/` folder. Paths are detected automatically; set `INPUT_DIR` and `INFLOW_PATH` below if needed. Outputs are three CSVs and matching metadata in a separate **`engineered_hydrology/`** folder.

| New group | Columns | Meaning |
|---|---:|---|
| Restored hourly inflow | 18 | Supplied inflow to each reservoir and the river, in m³/s |
| Local reference water balance | 9 | Initial storage plus natural inflow before the hour, minus current minimum storage and a reference estimate of prior water use, scaled to [-1, 1] |

The water-use proxy assumes full-plant reference discharge in earlier hours when price exceeds the plant's supplied terminal water value. Reference discharge comes from the largest turbine-curve discharge point for each unit in the supplied `Tokke_Vinje_topology.yaml`. Vatjern's missing initial volume uses its documented fixed 50% fill: 0.15 million m³. All other initial volumes come from the case inputs.

This is a simplified feature, not a simulated reservoir trajectory: it omits routed water, bypasses, head effects, and operating constraints. Only natural inflow to directly upstream reservoirs is summed. The denominator uses a stabilizer of 1 million m³. No labels, observed future storage changes, or generator-state history enter the calculation.

The existing 84 full-set features and nine relative price spreads are calculated directly from the original processed inputs; no earlier feature notebook needs to run first. Every calculation stays within one supplied 168-hour scheduling case. Hourly inflows are matched by UTC time and checked against the existing totals and weekly means, allowing for CSV rounding.

**0.995+ remains a leaderboard target.** The validation section records measured results; a new Kaggle submission is needed to establish the public score.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Config: set INPUT_DIR explicitly if the four original files are elsewhere.
INPUT_DIR = None
INFLOW_PATH = None  # Original hourly inflow CSV; auto-detected below.
HORIZON = 168
FILES = ("hourly_train.csv", "hourly_test.csv", "cases.csv", "meta.json")

if INPUT_DIR is None:
    INPUT_DIR = next((p for p in (Path("."), Path("processed"), Path("../processed"))
                      if all((p / name).is_file() for name in FILES)), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Upload all four preprocessed files together, or set INPUT_DIR.")
INPUT_DIR = Path(INPUT_DIR).resolve()
OUTPUT_DIR = (INPUT_DIR.parent if INPUT_DIR.name == "processed" else INPUT_DIR) / "engineered_hydrology"

meta = json.loads((INPUT_DIR / "meta.json").read_text())
if "feature_engineering" in meta:
    raise ValueError("Start from the original preprocessed files, not an engineered dataset.")
cases = pd.read_csv(INPUT_DIR / "cases.csv")
if cases["Run No"].isna().any() or cases["Run No"].duplicated().any():
    raise ValueError("cases.csv needs one non-missing Run No per case.")
base_hourly = [c for c in meta["hourly_feature_cols"] if not c.startswith("dvol_")]
base_static = [c for c in meta["static_feature_cols"] if not c.startswith("dvol_")]
removed = [c for c in cases if c.startswith("dvol_")]
print(f"Input: {INPUT_DIR}\nOutput: {OUTPUT_DIR}")
# Sum of the largest turbine-curve discharge points across each plant's units (m³/s).
# Fixed equipment data from the supplied Tokke_Vinje_topology.yaml; reference values,
# not head-dependent discharge limits. Vatjern's documented initial fill is 50% of 0.3 Mm³.
REFERENCE_FLOW = {"Hogga": 168.6, "Tokke": 128.0, "Lio": 14.0, "Byrte": 8.0,
                  "Vinje": 165.0, "Haukeli": 2.1, "Songa": 52.0,
                  "Kjela": 40.0, "Vesle Kjela": 20.0}

# Restore the supplied reservoir-specific hourly inputs.
inflow_name = "Historical_inflow_1958_2025.csv"
if INFLOW_PATH is None:
    locations = (INPUT_DIR, Path("."), Path("data/kernel"), Path("../data/kernel"),
                 INPUT_DIR.parent / "data/kernel")
    INFLOW_PATH = next((p / inflow_name for p in locations if (p / inflow_name).is_file()), None)
if INFLOW_PATH is None:
    raise FileNotFoundError(f"Upload {inflow_name} beside the preprocessed files, or set INFLOW_PATH.")
inflow_source = pd.read_csv(INFLOW_PATH)
inflow_source["date"] = pd.to_datetime(inflow_source["date"], utc=True)
inflow_source = inflow_source.set_index("date").sort_index()
if not inflow_source.index.is_unique:
    raise ValueError("The inflow CSV has duplicate timestamps.")
# Same hourly averaging as preprocessing; supplied data cover every required hour.
starts = pd.to_datetime(cases["starttime"], utc=True)
grid = pd.date_range(starts.min(), starts.max() + pd.Timedelta(hours=HORIZON - 1), freq="h")
inflow_source = inflow_source.resample("h").mean().reindex(grid)
if not np.isfinite(inflow_source.to_numpy()).all() or (inflow_source.to_numpy() < 0).any():
    raise ValueError("The raw inflow file must cover all case hours with finite, nonnegative inputs.")
print(f"Hourly inflow source: {Path(INFLOW_PATH).resolve()}")

Input: /Users/mjosefsen/Documents/NTNU/TDT4173/github/processed
Output: /Users/mjosefsen/Documents/NTNU/TDT4173/github/engineered_hydrology
Hourly inflow source: /Users/mjosefsen/Documents/NTNU/TDT4173/github/data/kernel/Historical_inflow_1958_2025.csv


## Feature calculations

All windows and cumulative sums stay inside one supplied 168-hour case. Targets are copied to the output but never used to calculate features.

In [2]:
# -----------------------------
# Feature calculations (pure functions; never read target columns)
# -----------------------------
def window_mean(a, width, forward):
    """Case × hour array; include the current hour and truncate at either edge."""
    a = np.asarray(a, dtype=np.float64)
    h = a.shape[1]
    t = np.arange(h)
    left = t if forward else np.maximum(0, t - width + 1)
    right = np.minimum(h, t + width) if forward else t + 1
    cumulative = np.pad(np.cumsum(a, axis=1), ((0, 0), (1, 0)))
    return (cumulative[:, right] - cumulative[:, left]) / (right - left)


def remaining_sum(a):
    """Sum from the current hour through the last hour, within each case."""
    return np.cumsum(np.asarray(a, dtype=np.float64)[:, ::-1], axis=1)[:, ::-1]


def block_total(values, active):
    """Sum values over the active block containing each hour; zero outside it."""
    left = np.zeros_like(values, dtype=float)
    right = np.zeros_like(left)
    for t in range(values.shape[1]):
        previous = left[:, t - 1] if t else 0
        left[:, t] = np.where(active[:, t], previous + values[:, t], 0)
    for t in range(values.shape[1] - 1, -1, -1):
        following = right[:, t + 1] if t + 1 < values.shape[1] else 0
        right[:, t] = np.where(active[:, t], following + values[:, t], 0)
    return left + right - np.where(active, values, 0)


def engineer_cases(source_cases, metadata):
    result = source_cases.drop(columns=[c for c in source_cases if c.startswith("dvol_")]).copy()
    new = {}
    for plant, mapping in metadata["plant_map"].items():
        side_fill = {}
        for side in ("up", "down"):
            columns = [f"fill_{r}" for r in mapping[side]]
            if columns and all(c in result for c in columns):
                side_fill[side] = result[columns].mean(axis=1).to_numpy()
                new[f"fe_initial_fill_{side}_{plant}"] = side_fill[side]
        if "up" in side_fill and "down" in side_fill:
            new[f"fe_initial_fill_up_minus_down_{plant}"] = side_fill["up"] - side_fill["down"]

    for column in source_cases:
        if not column.startswith("vol_"):
            continue
        reservoir = column[len("vol_"):]
        inflow_column = f"inflow_{reservoir}"
        if inflow_column not in result:
            continue
        initial = result[column].to_numpy(dtype=float)
        incoming = result[inflow_column].to_numpy(dtype=float) * HORIZON * 3600 / 1e6
        if (initial < 0).any() or (incoming < 0).any():
            raise ValueError("Inflow-share features require nonnegative storage and inflow.")
        denominator = initial + incoming
        new[f"fe_inflow_share_of_initial_plus_week_{reservoir}"] = np.divide(
            incoming, denominator, out=np.zeros_like(incoming), where=denominator > 0)
    additions = pd.DataFrame(new, index=result.index, dtype=np.float32)
    return pd.concat([result, additions], axis=1), list(additions.columns)


def engineer_hourly(source_hourly, source_cases, metadata, inflow_source):
    # Keep all window and block calculations inside each case.
    result = source_hourly.drop(columns=[c for c in source_hourly if c.startswith("dvol_")])
    result = result.sort_values(["Run No", "t"]).reset_index(drop=True)
    ids = result["Run No"].drop_duplicates().to_numpy()
    expected_t = np.tile(np.arange(HORIZON), len(ids))
    if (result["Run No"].isna().any() or not len(ids)
            or not result.groupby("Run No").size().eq(HORIZON).all()
            or not np.array_equal(result["t"].to_numpy(), expected_t)):
        raise ValueError("Each case must have one non-missing Run No and exactly t=0..167.")
    aligned = source_cases.set_index("Run No").loc[ids]
    shape = (len(ids), HORIZON)
    price = result["price"].to_numpy(dtype=float).reshape(shape)
    count_remaining = HORIZON - np.arange(HORIZON)
    new = {}
    for width in (6, 24):
        for direction, forward in (("past", False), ("next", True)):
            new[f"fe_price_minus_{direction}{width}h_mean"] = price - window_mean(price, width, forward)
    new["fe_remaining_price_max_minus_current"] = np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price
    new["fe_remaining_price_mean_minus_current"] = remaining_sum(price) / count_remaining - price

    extra = {}
    for plant in metadata["plant_map"]:
        spread = price - aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        above = spread > 0
        new[f"fe_price_minus_wvnet_{plant}"] = price - aligned[f"wvnet_{plant}"].to_numpy(dtype=float)[:, None]
        new[f"fe_above_wv_block_hours_{plant}"] = block_total(np.ones_like(spread), above)
        new[f"fe_remaining_above_wv_fraction_{plant}"] = remaining_sum(above) / count_remaining
        new[f"fe_remaining_positive_spread_mean_{plant}"] = remaining_sum(np.maximum(spread, 0)) / count_remaining
        water_value = aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        extra[f"fe_relative_price_spread_{plant}"] = spread / (np.abs(price) + np.abs(water_value) + 1.0)

    inflow = result["inflow_total_h"].to_numpy(dtype=float).reshape(shape)
    volume = inflow * 3600 / 1e6
    new["fe_inflow_before_hour_mm3"] = np.pad(np.cumsum(volume, axis=1)[:, :-1], ((0, 0), (1, 0)))
    new["fe_inflow_remaining_mm3"] = remaining_sum(volume)
    new["fe_inflow_minus_week_mean"] = inflow - inflow.mean(axis=1, keepdims=True)
    for column in result:
        if column.startswith("minvol_h_"):
            reservoir = column[len("minvol_h_"):]
            initial = aligned[f"vol_{reservoir}"].to_numpy(dtype=float)[:, None]
            new[f"fe_initial_volume_minus_minvol_{reservoir}"] = initial - result[column].to_numpy(dtype=float).reshape(shape)

    new.update(extra)

    # Align by UTC time, then verify against the aggregates already in the CSVs.
    starts = pd.DatetimeIndex(pd.to_datetime(aligned["starttime"], utc=True))
    positions = inflow_source.index.get_indexer(starts)
    if (positions < 0).any() or (positions + HORIZON > len(inflow_source)).any():
        raise ValueError("A case lies outside the supplied hourly inflow grid.")
    flow = inflow_source.to_numpy(dtype=np.float32)[positions[:, None] + np.arange(HORIZON)]
    totals = result["inflow_total_h"].to_numpy(dtype=np.float32).reshape(shape)
    means = aligned[[f"inflow_{r}" for r in inflow_source.columns]].to_numpy(dtype=float)
    # The original CSVs round to six significant digits.
    if (not np.allclose(flow.sum(axis=2), totals, rtol=1e-5, atol=2e-5)
            or not np.allclose(flow.mean(axis=1), means, rtol=1e-5, atol=2e-5)):
        raise ValueError("The inflow file does not match the preprocessed cases and hourly totals.")
    for j, reservoir in enumerate(inflow_source.columns):
        new[f"fe_inflow_hourly_{reservoir}"] = flow[:, :, j]

    local_supply = {
        plant: flow[:, :, [inflow_source.columns.get_loc(r) for r in mapping["up"]]].astype(float).sum(axis=2)
        for plant, mapping in metadata["plant_map"].items()
    }
    for plant, supply in local_supply.items():
        initial = np.zeros((len(ids), 1))
        minimum = np.zeros_like(supply)
        for reservoir in metadata["plant_map"][plant]["up"]:
            volume_column = f"vol_{reservoir}"
            if volume_column in aligned:
                initial += aligned[volume_column].to_numpy(dtype=float)[:, None]
            elif reservoir == "Vatjern":
                initial += 0.15  # Documented fixed initial storage, in million m³.
            else:
                raise ValueError(f"Missing initial storage for {reservoir}.")
            minimum_column = f"minvol_h_{reservoir}"
            if minimum_column in result:
                minimum += result[minimum_column].to_numpy(dtype=float).reshape(shape)
        water_value = aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        favorable_before = np.pad(np.cumsum(price > water_value, axis=1)[:, :-1], ((0, 0), (1, 0)))
        inflow_before = np.pad(np.cumsum(supply, axis=1)[:, :-1], ((0, 0), (1, 0))) * 3600 / 1e6
        supplied = initial + inflow_before - minimum
        reference_use = favorable_before * REFERENCE_FLOW[plant] * 3600 / 1e6
        new[f"fe_local_reference_water_balance_{plant}"] = (
            (supplied - reference_use) / (np.abs(supplied) + reference_use + 1.0)
        )
    additions = pd.DataFrame({name: a.ravel().astype(np.float32) for name, a in new.items()})
    return pd.concat([result, additions], axis=1), list(additions.columns)

In [3]:
# Build both datasets. Only the dataset and its metadata are exported.
engineered_cases, added_static = engineer_cases(cases, meta)
if not np.isfinite(engineered_cases[base_static + added_static].to_numpy(dtype=float)).all():
    raise ValueError("Missing/infinite case features; check cases.csv.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
added_hourly = None
for split in ("train", "test"):
    filename = f"hourly_{split}.csv"
    hourly = pd.read_csv(INPUT_DIR / filename)
    targets = meta["y_cols"] if split == "train" else []
    expected_columns = ["Run No"] + meta["hourly_feature_cols"] + targets
    expected_runs = set(cases.loc[cases["split"].eq(split), "Run No"])
    if list(hourly.columns) != expected_columns or set(hourly["Run No"]) != expected_runs:
        raise ValueError(f"{filename} does not match cases.csv / meta.json.")

    enriched, names = engineer_hourly(hourly, cases, meta, inflow_source)
    if added_hourly is not None and names != added_hourly:
        raise ValueError("Train and test feature columns differ.")
    added_hourly = names
    if not np.isfinite(enriched[base_hourly + names].to_numpy()).all():
        raise ValueError(f"Missing/infinite features in {filename}; check the input files.")
    enriched[["Run No"] + base_hourly + names + targets].to_csv(
        OUTPUT_DIR / filename, index=False, float_format="%.12g", chunksize=10000)
    print(f"Saved {filename}: {len(enriched):,} rows")
    del hourly, enriched

meta["hourly_feature_cols"] = base_hourly + added_hourly
meta["static_feature_cols"] = base_static + added_static
meta["use_end_of_week_volume"] = False
meta["feature_engineering"] = {
    "version": 4, "variant": "hydrology", "added_hourly_features": added_hourly,
    "added_static_features": added_static, "removed_features": removed,
    "horizon": HORIZON, "windows_include_current_hour": True, "uses_labels": False,
    "additional_input": inflow_name,
    "reference_flow_source": "data/extended/Tokke_Vinje_topology.yaml: turbine discharge curves",
    "reference_flow_m3_s": REFERENCE_FLOW,
    "fixed_initial_volume_mm3": {"Vatjern": 0.15},
    "balance_uses_hours_before_current": True,
    "new_groups": ['hourly_inflow', 'local_balance'],
    "new_hourly_features": [c for c in added_hourly if c.startswith(('fe_inflow_hourly_', 'fe_local_reference_water_balance_'))],

}
engineered_cases.to_csv(OUTPUT_DIR / "cases.csv", index=False, float_format="%.12g")
(OUTPUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
total = len(meta["hourly_feature_cols"]) + len(meta["static_feature_cols"])
print(f"Done: {len(added_hourly)} hourly + {len(added_static)} static additions; {total} total features.")

Saved hourly_train.csv: 490,896 rows
Saved hourly_test.csv: 121,800 rows
Done: 84 hourly + 36 static additions; 289 total features.


## Validation and measured results

The notebook was executed successfully on the complete dataset: **490,896 training rows (2,922 cases)** and **121,800 test rows (725 cases)**. All 262 existing scheduling features and all labels were preserved. The output has 289 finite input features, unique case/hour pairs, and exactly 168 ordered hours per case. Existing notebooks were left unchanged. An independent reviewer checked timestamp alignment, feature formulas, label independence, metadata, and Colab paths.

Feature selection used the original 300-tree XGBoost configuration: learning rate 0.05, depth 5, minimum child weight 5, row/column subsampling 0.8, L2 regularization 1, and seed 1337. Local runs used CPU. Training cases ended before 2022; validation cases started in 2022, with overlapping cases purged. All training rows were used.

The following comparison covers **six diagnostic units only**: Hogga G1, Tokke G2, Lio G1, Vinje G3, Haukeli G1, and Kjela G1. Micro ROC-AUC is calculated across their flattened predictions.

| Features | Count | Six-unit validation micro ROC-AUC |
|---|---:|---:|
| Existing scheduling dataset | 262 | 0.977766 |
| Scheduling + hourly inflows + cumulative/next-six-hour inflow summaries | 298 | 0.979427 |
| **Selected: scheduling + hourly inflows + local reference water balance** | **289** | **0.979747** |

For context, a separate completed **14-unit** comparison scored the scheduling baseline at **0.987331** and scheduling plus the 18 restored inflows alone at **0.987591**. These are different experiments from the six-unit table. The final 289-feature all-unit benchmark was stopped before completion to limit further work; its full score is not established.

These are development comparisons used to select features, rather than an untouched final evaluation. They show a promising direction, but do not establish improvement across other chronological folds or on Kaggle. The user-reported scheduling public score was **0.99053**; the new dataset has not been submitted. **0.995+ is not a measured or guaranteed result.** Keep the same model settings for the first full comparison and submission.


## Train with the new data

Use all four generated files from the same folder, including its `meta.json`. In your existing Colab training notebook, replace the automatic path assignment with:

```python
PROCESSED_PATH = Path("/content/engineered_hydrology")
```

Locally, use `Path("../engineered_hydrology")` from `github/notebooks/`. Rerun loading and training so metadata supplies the complete feature list. Start with the same XGBoost settings used for `engineered_scheduling` to compare fairly. The original inflow CSV is needed only to build these files, not for training or submission generation.

The separate original-data CV notebook has a guard that rejects engineered data. A copy adapted for engineered inputs must remove that restriction, retain the `dvol_*` exclusion, and use the matching generated metadata. Keep chronological case-level folds and their overlap gap.